# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze**, nettoyer les anomalies, typer fortement les colonnes (dates/timestamps, numériques) et créer vos colonnes métier dérivées.
3. Mettre en œuvre une **jointure distribuée optimisée** avec **Broadcast Hash Join** (`F.broadcast`) sur une table de référence (sans créer de produit cartésien N-M !).
4. Sauvegarder votre table analytique propre dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from delta.tables import DeltaTable

# Arrêt propre d'une éventuelle session résiduelle bloquée
try:
    spark.stop()
except:
    pass

# Initialisation combinée : Mémoire optimisée + Delta Lake actif
spark = (
    SparkSession.builder
    .appName("NYC_Taxi_Silver_Optimized")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

print("Session Spark initialisée avec succès (Mémoire optimisée + Delta Lake actif) — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e4a00319-27b5-4269-aea2-dd4877b7a102;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

Session Spark initialisée avec succès (Mémoire optimisée + Delta Lake actif) — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze & Diagnostic Qualité

### Exercice 1.1 — Profiler les valeurs manquantes, doublons et plages de valeurs
1. Chargez votre (ou vos) table(s) Delta depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` par colonne clé ainsi que le nombre de doublons éventuels sur l'identifiant métier.

In [2]:
# 1. Chargement de votre table Bronze principale depuis HDFS
BRONZE_MAIN_PATH = "hdfs://namenode:9000/lakehouse/bronze/nyc_taxi_delta"
df_bronze = spark.read.format("delta").load(BRONZE_MAIN_PATH)

assert df_bronze is not None, "TODO : Chargez votre table Bronze principale dans `df_bronze`."

# 2. Calcul du nombre de valeurs nulles par colonne
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns]
null_report = df_bronze.agg(*null_exprs)

assert null_report is not None, "TODO : Calculez le diagnostic de valeurs nulles `null_report`."
null_report.show(truncate=False)

# Affichage du nombre total de lignes Bronze
print(f"Nombre total de lignes Bronze : {df_bronze.count():,}")

26/11/18 10:01:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|ratecodeid|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|_ingested_at|_source_file|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+------------+
|0       |0                   |0                    |0              |0            

## 2. Nettoyage, Typage Fort & Jointure Optimisée (`BroadcastHashJoin`)

### Rappels d'architecture distribuée :
- **Attention aux jointures N-à-M :** Si vous joignez deux tables de détail partageant une clé non unique, pré-agrégez d'abord à la bonne granularité pour ne pas dupliquer les lignes et fausser vos sommes !
- **Broadcast Hash Join (`F.broadcast(petite_table)`) :** Lorsqu'une table de dimension / nomenclature fait quelques kilo-octets ou méga-octets, la diffuser sur chaque exécuteur supprime totalement l'étape coûteuse de **Shuffle** sur la grande table de faits.
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, enrichir par Broadcast Join et écrire `/lakehouse/silver/<table_silver>`
1. Supprimez les doublons (`dropDuplicates`), filtrez les valeurs techniquement impossibles (ex. durées ou montants négatifs) et convertissez explicitement vos dates (`F.to_timestamp` / `F.to_date`).
2. Créez au moins **2 nouvelles colonnes métier dérivées** utiles pour vos analyses des Jours 3 & 4 (ex. durée écoulée, écart, ratio, tranche horaire, mois `yyyy-MM`, indicateur binaire `0/1`...).
3. Réalisez une jointure avec `F.broadcast(...)` et affichez `.explain("simple")` pour prouver la présence de `BroadcastHashJoin`.
4. Écrivez la table résultante au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Nettoyage de sécurité de la session
try:
    spark.stop()
except:
    pass

# 2. Initialisation de la session Spark avec AQE activé (Adaptive Query Execution)
spark = (
    SparkSession.builder
    .appName("NYC_Taxi_Silver")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.memory.fraction", "0.5")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

SILVER_MAIN_PATH = "hdfs://namenode:9000/lakehouse/silver/nyc_taxi_silver"

print("1. Chargement de la table Bronze...")
df_bronze = spark.read.format("delta").load("hdfs://namenode:9000/lakehouse/bronze/nyc_taxi_delta")

print("2. Application des transformations (Nettoyage & Feature Engineering)...")
df_clean = (
    df_bronze
    .filter(F.col("trip_distance").isNotNull() & F.col("fare_amount").isNotNull())
    .withColumn("trip_distance_num", F.col("trip_distance").cast("double"))
    .withColumn("fare_amount_num", F.col("fare_amount").cast("double"))
    .filter((F.col("trip_distance_num") > 0.0) & (F.col("fare_amount_num") >= 0.0))
    .withColumn("pickup_datetime", F.to_timestamp("tpep_pickup_datetime"))
    .withColumn("dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime"))
    .withColumn("trip_month", F.date_format("pickup_datetime", "yyyy-MM"))
    .withColumn("is_night_trip", F.when((F.hour("pickup_datetime") >= 22) | (F.hour("pickup_datetime") < 6), 1).otherwise(0))
    .drop("trip_distance_num", "fare_amount_num")
)

print("3. Création de la table de référence des paiements et Broadcast...")
payment_types_data = [
    (1, "Credit card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip")
]
df_payment_ref = spark.createDataFrame(payment_types_data, ["payment_type", "payment_type_name"])

df_silver = df_clean.join(
    F.broadcast(df_payment_ref),
    on="payment_type",
    how="left"
)

print("4. Écriture sécurisée dans la couche Delta Silver (HDFS) avec coalesce(4)...")
(
    df_silver
    .coalesce(4)  # Réduit le nombre de partitions localement sans shuffle
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_MAIN_PATH)
)

print("Succès ! Écriture terminée. Lancement de la vérification...")

# 5. Lecture et affichage de l'échantillon
df_silver_check = spark.read.format("delta").load(SILVER_MAIN_PATH)
print("Succès ! La table Delta Silver est bien créée.")
df_silver_check.select("tpep_pickup_datetime", "trip_distance", "fare_amount", "payment_type_name", "is_night_trip").show(5, truncate=False)

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ad0ea7d5-a492-400f-8f74-c7639f4e8cda;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

1. Chargement de la table Bronze...
2. Application des transformations (Nettoyage & Feature Engineering)...
3. Création de la table de référence des paiements et Broadcast...
4. Écriture sécurisée dans la couche Delta Silver (HDFS) avec coalesce(4)...


26/11/18 10:09:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Succès ! Écriture terminée. Lancement de la vérification...
Succès ! La table Delta Silver est bien créée.
+--------------------+-------------+-----------+-----------------+-------------+
|tpep_pickup_datetime|trip_distance|fare_amount|payment_type_name|is_night_trip|
+--------------------+-------------+-----------+-----------------+-------------+
|2015-01-29 04:15:19 |.74          |5          |Cash             |1            |
|2015-01-29 04:15:19 |8.57         |26.5       |Cash             |1            |
|2015-01-29 04:15:19 |5.07         |16.5       |Cash             |1            |
|2015-01-29 04:15:26 |3.12         |14         |Credit card      |1            |
|2015-01-29 04:15:28 |.57          |4.5        |Credit card      |1            |
+--------------------+-------------+-----------+-----------------+-------------+
only showing top 5 rows


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [2]:
# 1. Lecture directe du journal transactionnel JSON sur HDFS
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# 2. Tester le Schema Enforcement avec un DataFrame au mauvais schéma dans un bloc try/except
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_MAIN_PATH)
except Exception as e:
    print("Schema Enforcement validé – écriture invalide bloquée :", str(e).splitlines()[0])

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794996706943,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"2528eab8-2ee8-4e2d-8f89-a266a15c37cf","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-a64c2500-0471-417b-afc2-f27704ce9f4f-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-bf3cf061-abd6-4655-803e-834fd73ae9a1-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-49f6e6a8-d7e8-48ee-913d-460bfa1de66f-c000.snappy.parquet","partition

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [1]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# 1. Initialisation de la session Spark
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("NYC_Taxi_Silver_Merge")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

SILVER_MAIN_PATH = "hdfs://namenode:9000/lakehouse/silver/nyc_taxi_silver"

# 2. Chargement de la table Delta existante
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)

# 3. On isole UNE seule ligne pour faire un upsert ultra-léger sans saturer la mémoire
df_sample = spark.read.format("delta").load(SILVER_MAIN_PATH).limit(1)

# On modifie le montant pour simuler la mise à jour
df_update = df_sample.withColumn("fare_amount", F.col("fare_amount") + 5.0)

# 4. Exécution sécurisée du MERGE INTO (Upsert)
target_delta.alias("target").merge(
    df_update.alias("source"),
    "target.tpep_pickup_datetime = source.tpep_pickup_datetime"
).whenMatchedUpdate(set={
    "fare_amount": "source.fare_amount"
}).whenNotMatchedInsertAll().execute()

print("Merge / Upsert exécuté avec succès !")

# 5. Affichage de l'historique des commits pour prouver la version >= 1
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)

# 6. Démonstration du Time Travel (version 0 vs version courante)
print("--- Échantillon de la Version 0 (Initiale) ---")
spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH).select("tpep_pickup_datetime", "fare_amount").show(3, truncate=False)

print("--- Échantillon de la Version Courante ---")
spark.read.format("delta").load(SILVER_MAIN_PATH).select("tpep_pickup_datetime", "fare_amount").show(3, truncate=False)

# Validation de l'exercice
assert history_df.count() >= 2, "TODO : Exécutez au moins une opération MERGE/UPDATE pour créer une version >= 1 !"
print("Exercice 3.2 validé avec succès ! 🎉")

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8d93f905-1c35-4407-b865-883b7ce86227;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

Merge / Upsert exécuté avec succès !
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      1|2026-11-18 10:22:18.682|    MERGE|{numTargetRowsCopied -> 11924057, numTargetRowsDeleted -> 0, numTargetBytesRe...|
|      0|2026-11-18 10:11:47.054|    WRITE|{numFiles -> 4, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> ...|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+

--- Échantillon de la Version 0 (Initiale) ---


+--------------------+-----------+
|tpep_pickup_datetime|fare_amount|
+--------------------+-----------+
|2015-01-29 04:15:19 |5          |
|2015-01-29 04:15:19 |26.5       |
|2015-01-29 04:15:19 |16.5       |
+--------------------+-----------+
only showing top 3 rows
--- Échantillon de la Version Courante ---
+--------------------+-----------+
|tpep_pickup_datetime|fare_amount|
+--------------------+-----------+
|2015-01-01 00:00:45 |16         |
|2015-01-01 00:00:54 |7          |
|2015-01-01 00:00:54 |5          |
+--------------------+-----------+
only showing top 3 rows
Exercice 3.2 validé avec succès ! 🎉
